In [1]:
import os
import sys

ROOT_PATH = '../../'
os.chdir(ROOT_PATH)

sys.path.append(os.path.abspath(ROOT_PATH))

lib_path = os.path.abspath(os.path.join(ROOT_PATH, 'lib'))
if lib_path not in sys.path:
  sys.path.append(lib_path)

In [2]:
import torch
from peft import LoraConfig, TaskType, get_peft_model
from ultralytics.utils.loss import v8SegmentationLoss, E2ELoss, BCEDiceLoss
from ultralytics.optim.muon import MuSGD
from torch.nn import L1Loss

from datasets.rescuenet_dataset import RescueNetDataset
from script.training.augmentations import get_augmentation_pipeline
from model.TriheadSegmentationModel import TriheadSegmentationModel
from script.final_model.UncertaintyLossWeighting import UncertaintyLossWeighting
from script.utils.training import compute_normal_loss, EarlyStoppingAndCheckpointing

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
triheadSegmentationModel = TriheadSegmentationModel(
  yolo_pt_path="yolo26m-seg.pt",
  device=device
)
target_modules = triheadSegmentationModel.yolo.modules()
loss_balancer = UncertaintyLossWeighting()

In [4]:
valid_target_modules = []
for name, module in triheadSegmentationModel.yolo.model.named_modules():
  if isinstance(module, torch.nn.Conv2d):
    if module.groups == 1:
      valid_target_modules.append(name)

In [5]:
lora_config = LoraConfig(
  r=16,
  lora_alpha=32,
  target_modules=valid_target_modules,
  modules_to_save=["model.23.*"],
  bias="none",
  task_type=TaskType.FEATURE_EXTRACTION,
)
peft_model = get_peft_model(triheadSegmentationModel.yolo.model, peft_config=lora_config).to(device)
peft_model.print_trainable_parameters()

trainable params: 1,368,496 || all params: 28,480,568 || trainable%: 4.8050


In [6]:
merged_pytorch_model: torch.nn.Module = peft_model.merge_and_unload()
merged_state_dict = merged_pytorch_model.state_dict()

In [7]:
spatial_aug, photometric_aug = get_augmentation_pipeline()

In [8]:
vanilla_train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', include_depth=False, spatial_transform=spatial_aug, photometric_transform=photometric_aug)
vanilla_val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val', include_depth=False)
vanilla_test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test', include_depth=False)
vanilla_train_loader = torch.utils.data.DataLoader(vanilla_train_dataset, batch_size=16, shuffle=True)
vanilla_val_loader = torch.utils.data.DataLoader  (vanilla_val_dataset, batch_size=16, shuffle=False)
vanilla_test_loader = torch.utils.data.DataLoader (vanilla_test_dataset, batch_size=16, shuffle=False)
print(f'Train Count: {len(vanilla_train_dataset)}')
print(f'Validation Count: {len(vanilla_val_dataset)}')
print(f'Test Count: {len(vanilla_test_dataset)}')

Train Count: 3595
Validation Count: 449
Test Count: 450


In [9]:
only_normals_train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', include_depth=False, spatial_transform=spatial_aug, photometric_transform=photometric_aug)
only_normals_val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val', include_depth=False)
only_normals_test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test', include_depth=False)
only_normals_train_loader = torch.utils.data.DataLoader(only_normals_train_dataset, batch_size=16, shuffle=True)
only_normals_val_loader = torch.utils.data.DataLoader(only_normals_val_dataset, batch_size=16, shuffle=False)
only_normals_test_loader = torch.utils.data.DataLoader(only_normals_test_dataset, batch_size=16, shuffle=False)
print(f'Train Count: {len(only_normals_train_dataset)}')
print(f'Validation Count: {len(only_normals_val_dataset)}')
print(f'Test Count: {len(only_normals_test_dataset)}')

Train Count: 3595
Validation Count: 449
Test Count: 450


In [10]:
only_depth_train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', include_normals=False, spatial_transform=spatial_aug, photometric_transform=photometric_aug)
only_depth_val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val', include_normals=False)
only_depth_test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test', include_normals=False)
only_depth_train_loader = torch.utils.data.DataLoader(only_depth_train_dataset, batch_size=16, shuffle=True)
only_depth_val_loader = torch.utils.data.DataLoader(only_depth_val_dataset, batch_size=16, shuffle=False)
only_depth_test_loader = torch.utils.data.DataLoader(only_depth_test_dataset, batch_size=16, shuffle=False)
print(f'Train Count: {len(only_depth_train_dataset)}')
print(f'Validation Count: {len(only_depth_val_dataset)}')
print(f'Test Count: {len(only_depth_test_dataset)}')

Train Count: 3595
Validation Count: 449
Test Count: 450


In [11]:
tri_train_dataset = RescueNetDataset(data_dir='./data/RescueNet/train', spatial_transform=spatial_aug, photometric_transform=photometric_aug)
tri_val_dataset = RescueNetDataset(data_dir='./data/RescueNet/val')
tri_test_dataset = RescueNetDataset(data_dir='./data/RescueNet/test')
tri_train_loader = torch.utils.data.DataLoader(tri_train_dataset, batch_size=16, shuffle=True)
tri_val_loader = torch.utils.data.DataLoader(tri_val_dataset, batch_size=16, shuffle=False)
tri_test_loader = torch.utils.data.DataLoader(tri_test_dataset, batch_size=16, shuffle=False)
print(f'Train Count: {len(tri_train_dataset)}')
print(f'Validation Count: {len(tri_val_dataset)}')
print(f'Test Count: {len(tri_test_dataset)}')

Train Count: 3595
Validation Count: 449
Test Count: 450


In [12]:
from types import SimpleNamespace

current_args = triheadSegmentationModel.yolo.model.args if isinstance(triheadSegmentationModel.yolo.model.args, dict) else {}

if 'overlap_mask' not in current_args:
  current_args['overlap_mask'] = True
  
triheadSegmentationModel.yolo.model.args = SimpleNamespace(**current_args)

In [13]:
musgd_params = []
sgd_params = []

for name, params in triheadSegmentationModel.named_parameters():
  if params.requires_grad:
    if params.ndim > 2:
      musgd_params.append(params)
    else:
      sgd_params.append(params)

sgd_params.extend(loss_balancer.parameters())

optimizer = MuSGD([
  { 'params': musgd_params },
  { 'params': sgd_params },
], lr=1e-3, momentum=0.937)

seg_loss_criterion = E2ELoss(triheadSegmentationModel.yolo.model, loss_fn=v8SegmentationLoss)
seg_loss_criterion.one2one.bce = BCEDiceLoss()
seg_loss_criterion.one2many.bce = BCEDiceLoss()
depth_loss_criterion = L1Loss()

In [14]:
# for batch_images, batch_targets in vanilla_train_loader:
#   break
#   # optimizer.zero_grad()
  
#   # segmentation_out, depth_out, normal_out = triheadSegmentationModel(batch_images)
  
#   # true_segmentation_map = batch_targets['segmentation']
#   # true_depth_map = batch_targets['depth_map']
#   # true_surface_normals = batch_targets['surface_normals']

#   # seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
#   # depth_loss = depth_loss_criterion(depth_out, true_depth_map)
#   # normal_loss = compute_normal_loss(normal_out, true_surface_normals)
  
#   # loss_total = (torch.abs(loss_balancer.alpha) * seg_loss) +(torch.abs(loss_balancer.beta) * depth_loss) + (torch.abs(loss_balancer.gamma) * normal_loss)
  
#   # loss_total.backward()
#   # optimizer.step()

In [15]:
# for batch_images, batch_targets in only_depth_train_loader:
#   break
#   # optimizer.zero_grad()
  
#   # segmentation_out, depth_out, normal_out = triheadSegmentationModel(batch_images)
  
#   # true_segmentation_map = batch_targets['segmentation']
#   # true_depth_map = batch_targets['depth_map']
#   # true_surface_normals = batch_targets['surface_normals']

#   # seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
#   # depth_loss = depth_loss_criterion(depth_out, true_depth_map)
#   # normal_loss = compute_normal_loss(normal_out, true_surface_normals)
  
#   # loss_total = (torch.abs(loss_balancer.alpha) * seg_loss) +(torch.abs(loss_balancer.beta) * depth_loss) + (torch.abs(loss_balancer.gamma) * normal_loss)
  
#   # loss_total.backward()
#   # optimizer.step()

In [16]:
# for batch_images, batch_targets in only_normals_train_loader:
#   break
#   # optimizer.zero_grad()
  
#   # segmentation_out, depth_out, normal_out = triheadSegmentationModel(batch_images)
  
#   # true_segmentation_map = batch_targets['segmentation']
#   # true_depth_map = batch_targets['depth_map']
#   # true_surface_normals = batch_targets['surface_normals']

#   # seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
#   # depth_loss = depth_loss_criterion(depth_out, true_depth_map)
#   # normal_loss = compute_normal_loss(normal_out, true_surface_normals)
  
#   # loss_total = (torch.abs(loss_balancer.alpha) * seg_loss) +(torch.abs(loss_balancer.beta) * depth_loss) + (torch.abs(loss_balancer.gamma) * normal_loss)
  
#   # loss_total.backward()
#   # optimizer.step()

In [18]:
early_stopping = EarlyStoppingAndCheckpointing()

for batch_images, batch_targets in tri_train_loader:
  optimizer.zero_grad()
  
  segmentation_out, depth_out, normal_out = triheadSegmentationModel.forward(batch_images.to(device=device))
  
  true_segmentation_map = batch_targets['segmentation']
  true_depth_map = batch_targets['depth_map']
  true_surface_normals = batch_targets['surface_normals']
  proto_tensor = segmentation_out[1]
  print(true_segmentation_map)
  
  seg_loss, seg_loss_items = seg_loss_criterion(segmentation_out, true_segmentation_map)
  
  depth_loss = depth_loss_criterion(depth_out, true_depth_map)
  normal_loss = compute_normal_loss(normal_out, true_surface_normals)
  
  loss_total = (torch.abs(loss_balancer.alpha) * seg_loss) +(torch.abs(loss_balancer.beta) * depth_loss) + (torch.abs(loss_balancer.gamma) * normal_loss)
  
  loss_total.backward()
  optimizer.step()
  break

tensor([[[ 9,  9,  9,  ...,  9,  9,  9],
         [ 9,  9,  9,  ...,  9,  9,  9],
         [ 9,  9,  9,  ...,  9,  9,  9],
         ...,
         [ 9,  9,  9,  ...,  9,  9,  9],
         [ 9,  9,  9,  ...,  9,  9,  9],
         [ 9,  9,  9,  ...,  9,  9,  9]],

        [[11, 11, 11,  ..., 11, 11, 11],
         [11, 11, 11,  ..., 11, 11, 11],
         [11, 11, 11,  ..., 11, 11, 11],
         ...,
         [ 1,  1,  1,  ...,  9,  9,  9],
         [ 1,  1,  1,  ...,  9,  9,  9],
         [ 1,  1,  1,  ...,  9,  9,  9]],

        [[ 0,  0,  0,  ...,  0,  0,  0],
         [ 0,  0,  0,  ...,  0,  0,  0],
         [ 0,  0,  0,  ...,  0,  0,  0],
         ...,
         [ 0,  0,  0,  ...,  0,  0,  0],
         [ 0,  0,  0,  ...,  0,  0,  0],
         [ 0,  0,  0,  ...,  0,  0,  0]],

        ...,

        [[ 1,  1,  1,  ...,  1,  1,  1],
         [ 1,  1,  1,  ...,  1,  1,  1],
         [ 1,  1,  1,  ...,  1,  1,  1],
         ...,
         [ 1,  1,  1,  ...,  1,  1,  1],
         [ 1,  1,  1, 

IndexError: too many indices for tensor of dimension 3